# Gradio

### Let's make a User Interface for the chatbot!

---

So far, we have called language models directly from Python code.

Now we will use **Gradio** to turn our model into a small interactive web application.

Gradio is a Python library that helps us quickly build user interfaces for machine learning and AI applications. Instead of only running code cells, we can create an interface where users type input, click buttons, and see model responses.

The basic idea is:

```text
User input → Gradio interface → Python function → AI model → Response shown in the UI
```

In this notebook, we will build a simple chatbot interface and connect it to our model function.

---

#### Let start very simple:

In [ ]:
import gradio as gr

In [ ]:
def greeting(name):
    return f"Hello {name}!"

Now I have a very simple function, I can give a name as an imput, call it, and see the output:

In [ ]:
greeting("Aram")

In this step, I want to create a Gradio interface that allows me to call this function.

In [ ]:
app = gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text"
)

app.launch()

---

### Sharing a Gradio app

When we launch a Gradio app, it usually runs locally on our own machine.

By setting `share=True`, Gradio creates a temporary public link that allows other people to open and interact with our app from their own browser.

What happens behind the scenes is:

Other user → Public Gradio link → Tunnel to your computer → Your Python function → Output returned to the interface


So the app is still running on **your machine**.
Gradio only provides a temporary public connection to it.

This is very useful for demos, testing, and sharing prototypes with others.

In [ ]:
app.launch(share = True)

Let's make it a bit fancy! :)

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=greeting,
    title="Greeting", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

---

### From a simple function to an AI function

Yes! This is the important idea.

In the first Gradio example, we may call a simple Python function like `greeting()`.

But in a real AI application, the function can do something more powerful:

```text
User input → Gradio interface → chat function → LLM API/local model → response
```

So instead of returning a fixed message, our function sends the user input to an LLM and returns the model’s answer.

This means Gradio is not the AI model itself.
Gradio is the interface that connects the user to our Python function, and our Python function connects to the model.

Let's try to have our very first chatbot interface! (Use any local/foundation model that is available for you!)

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

openai = OpenAI()
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

    
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

---

### Making the chatbot feel more real

As you can see, our current app waits until the model has generated the **full response**.

Only when the answer is completely ready, Gradio shows it in the output box.

This works, but it does not feel like the chatbots we usually use, such as ChatGPT, where the answer appears gradually while the model is generating it.

So now we will rewrite our chat function to use **streaming**.

With streaming, the model sends small pieces of the answer step by step, and our interface updates while the response is still being generated.

The idea is:

```text
User message → Model starts generating → Show partial answer → Keep updating → Final answer
```

This makes the chatbot feel more natural and interactive.

In [ ]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [ ]:
system_message = "You are a helpful assistant"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

---

## Understanding `gr.Interface`

---

The `Interface` class is the simplest way to build a web application with Gradio.

A Gradio interface connects:

1. A Python function
2. Input components
3. Output components

### How it works?

```text
User Input
     │
     ▼
Input Component
     │
     ▼
Python Function (fn)
     │
     ▼
Output Component
     │
     ▼
Displayed Result
```

When the user interacts with the webpage:

1. Gradio collects values from the input components.
2. These values are passed to the Python function.
3. The function executes.
4. The return value is sent to the output components.
5. The result is displayed in the browser.

---


## Main parameters of `gr.Interface`

### `fn`

The Python function that will be executed. Like the `greeting` fuction we had before.

The number of inputs should match the number of function parameters.

---

### `inputs`

Defines what the user can provide.

#### Simple form

```python
inputs="text"
```

Equivalent to:

```python
inputs=gr.Textbox()
```

#### Multiple inputs

```python
inputs=[
    gr.Number(),
    gr.Number()
]
```
---

### `outputs`

Defines how the result is displayed.

#### Simple output

```python
outputs="text"
```

Equivalent to:

```python
outputs=gr.Textbox()
```

#### Multiple outputs

```python
outputs=[
    gr.Textbox(),
    gr.Number()
]
```

The function must return multiple values:

```python
def analyze(text):
    return text.upper(), len(text)
```

---

### `title`

Adds a title to the web application.

In [ ]:
UI = gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text",
    title="My First Gradio App"
)

#UI.launch()

---

### `description`

Adds explanatory text below the title.

In [ ]:

UI = gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text",
    title="Greeting App",
    description="Enter your name and receive a greeting."
)

#UI.launch()

---

### `examples`

Provides sample inputs that users can click.


In [ ]:
UI = gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text",
    examples=[
        ["Alice"],
        ["Bob"],
        ["Charlie"]
    ]
)

#UI.launch()


---

### `allow_flagging`

Controls whether users can report problematic outputs.

```python
allow_flagging="never"
```

Options:

- `"never"`
- `"manual"`
- `"auto"`

Example:

```python
gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text",
    allow_flagging="never"
)
```


---

### `theme`

Changes the appearance of the application.

```python
theme="soft"
```


In [ ]:
UI = gr.Interface(
    fn=greeting,
    inputs="text",
    outputs="text",
    theme="soft"
)

#UI.launch()


---

### `live`

Automatically runs the function whenever the input changes.

```python
live=True
```

In [ ]:
UI = gr.Interface(
    fn=lambda x: x.upper(),
    inputs="text",
    outputs="text",
    live=True
)

#UI.launch()

Without pressing a submit button, the output updates instantly.

---

### `submit_btn`

Customize the submit button text.

```python
submit_btn="Analyze"
```

---

### `clear_btn`

Customize the clear button text.

```python
clear_btn="Reset"
```

---

## Common Input Components

| Component | Description |
|------------|-------------|
| `gr.Textbox()` | Text input |
| `gr.Number()` | Numeric input |
| `gr.Slider()` | Slider |
| `gr.Dropdown()` | Dropdown menu |
| `gr.Radio()` | Radio buttons |
| `gr.Checkbox()` | Checkbox |
| `gr.Image()` | Image upload |
| `gr.Audio()` | Audio upload |
| `gr.Video()` | Video upload |
| `gr.File()` | File upload |

---

## Common Output Components

| Component | Description |
|------------|-------------|
| `gr.Textbox()` | Text output |
| `gr.Number()` | Numeric output |
| `gr.Label()` | Classification labels |
| `gr.Image()` | Display image |
| `gr.Audio()` | Play audio |
| `gr.Video()` | Display video |
| `gr.Dataframe()` | Display table |
| `gr.JSON()` | Display JSON data |
| `gr.Plot()` | Display charts |

Example:

```python
outputs=gr.Label()
```

Useful for machine learning classification results.

---


## Complete Example

In [ ]:
import gradio as gr

def classify_review(review):
    if "good" in review.lower():
        return "Positive"
    return "Negative"

app = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(
        lines=5,
        label="Review"
    ),
    outputs=gr.Label(),
    title="Review Classifier",
    description="Classify a review as positive or negative.",
    examples=[
        ["This course was very good."],
        ["I did not enjoy this course."]
    ]
)

app.launch()



---

## Key Takeaway

A Gradio interface is simply:

```text
Input Components
        ↓
Python Function
        ↓
Output Components
```

Gradio automatically generates the web application, allowing Python developers to build interactive AI and data applications without writing HTML, CSS, or JavaScript.

## Exercise: Build a Gradio UI for the LLM Debate

In the previous notebook, you built a small program where two LLMs talk to each other.

Now your task is to turn that program into a small **interactive Gradio app**.

The goal is to build a UI where the user can:

1. choose or edit the two chatbot personalities,
2. choose the debate topic,
3. choose the number of rounds,
4. start the debate,
5. see the conversation between the two models in the interface.

This exercise connects three important ideas:

```text
System messages + Conversation history + Gradio interface
```
 ---


#### Your Task

Your app should have inputs for:

- Character A name
- Character B name
- Character A system message
- Character B system message
- Debate topic
- Number of rounds

Your app should return:

- the full debate conversation as text, or
- a chat-style output showing the messages from both models.

Start simple first.  
Make it work with text output, then improve the design.

---

### **Challenge Ideas**

After your first version works, improve the app.

Try at least two:

1. Add a dropdown to choose the OpenAI model.
2. Add a dropdown to choose the Ollama model.
3. Add a `share=True` launch option and test the app with a classmate.
4. Change the output from a text box to a chat-style output.
5. Add a third model as a judge.
6. Add a button to clear the debate.
7. Add example topics using `gr.Examples`.
8. Add a warning when the number of rounds is too high.

Remember:

```text
A good AI app is not only about the model.
It is also about the interface, user control, safety, and experience.
```